In [ ]:
import re
import json
import csv

# خواندن فایل
with open("questions.txt", "r", encoding="utf-8") as f:
    content = f.read()

# نرمالسازی ارقام فارسی/عربی به لاتین
def normalize_digits(text):
    persian = "۰۱۲۳۴۵۶۷۸۹"
    arabic = "٠١٢٣٤٥٦٧٨٩"
    latin = "0123456789"
    trans = str.maketrans(persian + arabic, latin * 2)
    return text.translate(trans)

content = normalize_digits(content)
lines = content.split("\n")

category = None
questions = []
current_q = None
current_opts = []

for line in lines:
    line = line.strip()
    if not line:
        continue
    
    # تشخیص سوال جدید
    q_match = re.match(r"^(\d+)\s*[-\.):]?\s*(.+)$", line)
    opt_match = re.match(r"^\(?([1-4])\)?[\)\.\-:]\s*(.+)$", line)
    
    if q_match and not opt_match:
        qnum = int(q_match.group(1))
        
        # ذخیره سوال قبلی
        if current_q and 1 <= int(current_q["number"]) <= 140:
            questions.append({
                "question_number": current_q["number"],
                "category": current_q["category"],
                "question": current_q["text"],
                "options": current_opts
            })
        
        # شروع سوال جدید
        if 1 <= qnum <= 140:
            current_q = {
                "number": str(qnum),
                "category": category if category else "نامشخص",
                "text": q_match.group(2).strip()
            }
            current_opts = []
        else:
            category = line
            current_q = None
    
    elif opt_match and current_q:
        opt_num = opt_match.group(1)
        opt_text = opt_match.group(2).strip()
        current_opts.append(f"{opt_num}) {opt_text}")
    
    elif current_q and not opt_match and not q_match:
        if not current_opts:
            current_q["text"] += " " + line
        else:
            if current_opts:
                current_opts[-1] += " " + line
    
    elif not q_match and not opt_match and not current_q:
        category = line

# ذخیره آخرین سوال
if current_q and 1 <= int(current_q["number"]) <= 140:
    questions.append({
        "question_number": current_q["number"],
        "category": current_q["category"],
        "question": current_q["text"],
        "options": current_opts
    })

# مرتب‌سازی بر اساس شماره سوال
questions.sort(key=lambda x: int(x["question_number"]))

# ذخیره JSON
with open("structured_questions.json", "w", encoding="utf-8") as jf:
    json.dump(questions, jf, ensure_ascii=False, indent=2)

# ذخیره CSV
with open("structured_questions.csv", "w", encoding="utf-8-sig", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=["question_number", "category", "question", "options"])
    writer.writeheader()
    for q in questions:
        writer.writerow({
            "question_number": q["question_number"],
            "category": q["category"],
            "question": q["question"],
            "options": " | ".join(q["options"])
        })

print(f"✓ تعداد سوالات: {len(questions)}")
print(f"✓ فایل‌ها ذخیره شدند: structured_questions.json, structured_questions.csv")


✓ تعداد سوالات: 120
✓ فایل‌ها ذخیره شدند: structured_questions.json, structured_questions.csv
